## Análisis de los niveles de O3 en Montevideo
### Para esto, se utilizaron los datasets publicos dispuestos por la Intendencia Municipal de Montevideo. 
![Imagen_de_la_imm](https://www.gub.uy/catalogo-participacion-ciudadana/sites/catalogo-participacion-ciudadana/files/styles/documento/public/2024-05/Logo%20Intendencia%20de%20Montevideo_3.png?itok=dmOKXdGC)

### Información sobre el dataset:
#### Rango de fechas: 2024-01-01 hasta 2026-09-19 (desde el primer hasta el último registro)
#### Estaciones de control que miden el O3: "Maroñas" y "Colón", las demás miden NO2 o PM 2.5
#### Cantidad de regristros: 2.188.716
#### Cantidad de columnas: 6
#### Cantidad de regristros nulos: 373.205

## Importación de los datos

In [84]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import seaborn as sns
import matplotlib.pyplot as plt

In [85]:
BASE_URL = "https://ckan.montevideo.gub.uy"
DATASET_ID_O3 = "calidad-del-aire-ozono-o3"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

r_o3 = requests.get(f"{BASE_URL}/api/3/action/package_show", params={"id": DATASET_ID_O3}, headers=headers)
resources_o3 = r_o3.json()["result"]["resources"]

for res in resources_o3:
    print(res["name"], "-", res["format"], "-", res["url"])

dfs_o3 = []
for res in resources_o3:
    if res["format"].upper() == "CSV":
        try:
            df = pd.read_csv(res["url"], storage_options=headers)
            df["archivo_origen"] = res["name"]
            dfs_o3.append(df)
        except Exception as e:
            print(f"Error con {res['name']}: {e}")

o3_completo = pd.concat(dfs_o3, ignore_index=True)

O3  01-2024 - 04-2024 - CSV - https://ckan-data.montevideo.gub.uy/dataset/70a1a63d-4157-43c0-9ae6-00fc14dae29a/resource/a97d7ffb-be66-4438-8de8-fa4c7cacf66c/download/o3_01_2024_04_2024.csv
O3  05-2024 - 08-2024 - CSV - https://ckan-data.montevideo.gub.uy/dataset/70a1a63d-4157-43c0-9ae6-00fc14dae29a/resource/e5d34ea0-6f21-410c-9085-bf913fbdede8/download/o3_05_2024_08_2024.csv
O3  09-2024 - 12-2024 - CSV - https://ckan-data.montevideo.gub.uy/dataset/70a1a63d-4157-43c0-9ae6-00fc14dae29a/resource/7f650358-e61a-434c-8953-e2c08ca0f738/download/o3_09_2024_12_2024.csv
O3  01-2025 - 04-2025 - CSV - https://ckan-data.montevideo.gub.uy/dataset/70a1a63d-4157-43c0-9ae6-00fc14dae29a/resource/a33889b6-a121-415d-aa9c-907c28ee7bfd/download/o3_01_2025_04_2025.csv
O3  05-2025 - 08-2025 - CSV - https://ckan-data.montevideo.gub.uy/dataset/70a1a63d-4157-43c0-9ae6-00fc14dae29a/resource/31628cd8-7222-4c69-b017-6584f0eaef75/download/o3_05_2025_08_2025.csv
O3  09-2025 - 12-2025 - CSV - https://ckan-data.montevi

## Visualización primaria de los datos

In [86]:
o3_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2188716 entries, 0 to 2188715
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   fecha           object 
 1   o3              float64
 2   estacion        object 
 3   longitud        float64
 4   latitud         float64
 5   archivo_origen  object 
dtypes: float64(3), object(3)
memory usage: 100.2+ MB


In [87]:
o3_completo.shape

(2188716, 6)

In [88]:
o3_completo.head()

,fecha,o3,estacion,longitud,latitud,archivo_origen
0,2024-01-01 00:00:00,6.0,Colón,-56.224168,-34.798337,O3 01-2024 - 04-2024
1,2024-01-01 00:00:00,NaN,Curva de Maroñas,-56.133249,-34.858960,O3 01-2024 - 04-2024
2,2024-01-01 00:01:00,8.0,Colón,-56.224168,-34.798337,O3 01-2024 - 04-2024
3,2024-01-01 00:01:00,NaN,Curva de Maroñas,-56.133249,-34.858960,O3 01-2024 - 04-2024
4,2024-01-01 00:02:00,8.0,Colón,-56.224168,-34.798337,O3 01-2024 - 04-2024


In [89]:
o3_completo.tail()

,fecha,o3,estacion,longitud,latitud,archivo_origen
2188711,2026-09-19 23:55:00,45.0,Colón,-56.224168,-34.798337,O3 09-2026 - 12-2026
2188712,2026-09-19 23:56:00,45.0,Colón,-56.224168,-34.798337,O3 09-2026 - 12-2026
2188713,2026-09-19 23:57:00,43.0,Colón,-56.224168,-34.798337,O3 09-2026 - 12-2026
2188714,2026-09-19 23:58:00,43.0,Colón,-56.224168,-34.798337,O3 09-2026 - 12-2026
2188715,2026-09-19 23:59:00,49.0,Colón,-56.224168,-34.798337,O3 09-2026 - 12-2026


In [90]:
o3_completo.isna().sum()

fecha                  0
o3                373205
estacion               0
longitud               0
latitud                0
archivo_origen         0
dtype: int64

In [91]:
print(f"---------------------------------")
print(o3_completo["estacion"].value_counts())
print(f"The amount of entries are: //// {o3_completo['estacion'].value_counts().sum()} ////")
print(f"---------------------------------")
print(o3_completo.groupby("estacion")["o3"].apply(lambda x: x.isna().mean() * 100).round(1))
print(f"---------------------------------")
print(o3_completo.groupby("archivo_origen")["o3"].apply(lambda x: x.isna().mean() * 100).round(1))
print(f"---------------------------------")

---------------------------------
estacion
Curva de Maroñas    1273297
Colón                915419
Name: count, dtype: int64
The amount of entries are: //// 2188716 ////
---------------------------------
estacion
Colón               19.0
Curva de Maroñas    15.7
Name: o3, dtype: float64
---------------------------------
archivo_origen
O3  01-2024 - 04-2024    12.4
O3  01-2025 - 04-2025     2.8
O3  01-2026 - 04-2026     1.6
O3  05-2024 - 08-2024    14.2
O3  05-2025 - 08-2025    26.1
O3  05-2026 - 08-2026    21.3
O3  09-2024 - 12-2024     3.6
O3  09-2025 - 12-2025    39.5
O3  09-2026 - 12-2026    31.3
Name: o3, dtype: float64
---------------------------------


## EDA

In [92]:
# Antes que nada, se registra las filas antes de limpiar
fa = len(o3_completo) # fa = filas_antes

# Se limpian los NaN de la columna O3
df_purgado = o3_completo.dropna(subset=["o3"]).copy()

# Se registra las filas después de limpiar
fd = len(df_purgado) # fd = filas_después

filas_descartadas = fa - fd
porcentaje_conservado = (fd / fa) * 100

print(f"Filas originales:     {fa:,}")
print(f"Filas conservadas:    {fd:,}")
print(f"Filas descartadas:    {filas_descartadas:,}")
print(f"Porcentaje restante:  {porcentaje_conservado:.2f}%")

Filas originales:     2,188,716
Filas conservadas:    1,815,511
Filas descartadas:    373,205
Porcentaje restante:  82.95%


In [93]:
df_purgado.dtypes 
# Hay que convertir la columna de "fecha", la cual es "object" en datetime

fecha              object
o3                float64
estacion           object
longitud          float64
latitud           float64
archivo_origen     object
dtype: object

In [94]:
# Convertir la columna 'fecha' de texto (object) a datetime real de pandas
df_purgado['fecha'] = pd.to_datetime(df_purgado['fecha'])

# Extraer la hora de la comuna fecha, crear su propia columna
df_purgado['hora_de_la_muestra'] = df_purgado['fecha'].dt.hour

# Crear columna de día de semana (nombre en inglés: Monday, Tuesday, etc.)
df_purgado['dia_semana'] = df_purgado['fecha'].dt.day_name()

# Crear columna categórica simplificada: día de semana vs. fin de semana
df_purgado['tipo_dia'] = df_purgado['fecha'].dt.dayofweek.apply(lambda x: "Fin de semana" if x >= 5 else "Día de semana")

df_purgado = df_purgado.reset_index(drop=True)
print(f"---------------------------------")
print(df_purgado["estacion"].value_counts())
print(f"---------------------------------")
print(f"La cantidad de entradas son: //// {df_purgado['estacion'].value_counts().sum()} ////")
print(f"Estación 'Curva de Maroñas' - % de datos restante: {round((199515 / 1273297) * 100, 1)}%")
print(f"Estación 'Colón' - % de datos restante: {round((172250 / 913979) * 100, 1)}%")

---------------------------------
estacion
Curva de Maroñas    1073782
Colón                741729
Name: count, dtype: int64
---------------------------------
La cantidad de entradas son: //// 1815511 ////
Estación 'Curva de Maroñas' - % de datos restante: 15.7%
Estación 'Colón' - % de datos restante: 18.8%


In [95]:
df_purgado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1815511 entries, 0 to 1815510
Data columns (total 9 columns):
 #   Column              Dtype         
---  ------              -----         
 0   fecha               datetime64[ns]
 1   o3                  float64       
 2   estacion            object        
 3   longitud            float64       
 4   latitud             float64       
 5   archivo_origen      object        
 6   hora_de_la_muestra  int32         
 7   dia_semana          object        
 8   tipo_dia            object        
dtypes: datetime64[ns](1), float64(3), int32(1), object(4)
memory usage: 117.7+ MB


## Exportación a CSV intermedio
### A continuación se exportará el dataset como resultado del Exploratory Data Analysis

In [97]:
df_purgado.to_csv("Dataset_O3_listo.csv", index=False)

## Decisiones tomadas sobre la limpieza de datos

### 1. Cambio de contaminante: de NO2 a O3

Inicialmente se trabajó con el dataset de Dióxido de Nitrógeno (NO2), pero se detectó que ~70% 
de sus valores eran nulos, distribuidos en bloques trimestrales completos (varios trimestres al 
100% de nulos) — un patrón compatible con un problema de publicación del dataset, no con fallas 
normales de sensor. Se envió una consulta a datosabiertos@imm.gub.uy para confirmar la causa 
(respuesta pendiente).

Se optó por continuar el análisis con Ozono (O3), que presenta ~17% de nulos distribuidos de 
forma gradual por trimestre (entre 1.6% y 39.5%, sin bloques en 0% o 100%), consistente con 
interrupciones normales de sensor y apto para el análisis.

### 2. Tratamiento de valores nulos en la columna 'o3'

El dataset de O3 contenía 373.205 filas con valores nulos en la columna 'o3' (17.05% del total 
de 2.188.716 filas). Se optó por eliminar estas filas (dropna) en lugar de imputarlas 
(por ejemplo, con media o mediana), por dos razones:

1. El volumen de nulos era demasiado alto (17%) como para imputar sin distorsionar la varianza 
   real de los datos — rellenar esa proporción con un único valor central achataría 
   artificialmente la distribución, afectando directamente los análisis estadísticos posteriores 
   (Día 4: tests de inferencia, que dependen de la varianza real).
2. La columna 'o3' es la variable de interés central del análisis — no tiene sentido inventar 
   el propio dato que se busca estudiar.

Resultado: 1.815.511 filas conservadas (82.95% del total).

### 3. Verificación de sesgo por estación tras la limpieza

Se verificó que la eliminación de nulos no afectara desproporcionadamente a una de las dos 
estaciones disponibles (Colón y Curva de Maroñas):

- Curva de Maroñas: perdió 15.67% de sus registros originales.
- Colón: perdió 18.85% de sus registros originales.

La diferencia entre ambas (≈3 puntos porcentuales) se considera menor y no introduce un sesgo 
relevante para la comparación entre estaciones planeada en el análisis inferencial.

### 4. Reseteo del índice

Se reinició el índice del DataFrame tras la eliminación de filas, por motivos de 
legibilidad: pandas no renumera automáticamente el índice al eliminar filas, dejando huecos en 
la numeración. Esto no afecta los cálculos estadísticos ni la performance, es únicamente una 
decisión de prolijidad.

### 5. Columnas derivadas de fecha

A partir de la columna 'fecha' (convertida de texto a datetime) se derivaron tres columnas 
nuevas: 'hora_de_la_muestra' (0-23, para analizar variación horaria del O3), 'dia_semana' 
(nombre del día) y 'tipo_dia' (Día de semana / Fin de semana, agrupando sábado y domingo).

### 6. Exportación

El dataset resultante, limpio y con las columnas derivadas, se exportó como archivo CSV 
intermedio para continuar el análisis exploratorio sin necesidad de repetir la 
descarga y limpieza en cada sesión de trabajo.